In [1]:
import numpy as np
import pandas as pd


In [ ]:
#1.1
df = pd.read_csv('data/purchases.csv')
customer_info = pd.read_csv("data/customer_info.csv")
df = df.sample(frac=1, random_state=42)


In [6]:
print(df.head())
print(customer_info.head())

                    Customer_id   Price
Timestamp                              
2025-01-01 00:00:00        C037  436.71
2025-01-01 00:30:00        C049  274.70
2025-01-01 01:00:00        C017  380.42
2025-01-01 01:30:00        C049  378.28
2025-01-01 02:00:00        C002   13.88
  Customer_id   Born_date       City
0        C001  1998-12-03  Frankfurt
1        C002  1988-07-03  Frankfurt
2        C003  1974-12-01     Munich
3        C004  2002-08-27  Frankfurt
4        C005  1967-03-05    Cologne


In [ ]:
#1.2
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

# 1.3 
df = df.set_index('Timestamp').sort_index()
print(df.head())



                    Customer_id   Price
Timestamp                              
2025-01-01 00:00:00        C037  436.71
2025-01-01 00:30:00        C049  274.70
2025-01-01 01:00:00        C017  380.42
2025-01-01 01:30:00        C049  378.28
2025-01-01 02:00:00        C002   13.88


In [ ]:
#2.1, 2.2
merged = df.reset_index().merge(customer_info[['Customer_id', 'Born_date', 'City']], on='Customer_id', how='left')
merged = merged.set_index('Timestamp').sort_index()
print(merged.head())


                    Customer_id   Price   Born_date       City
Timestamp                                                     
2025-01-01 00:00:00        C037  436.71  1996-09-14     Munich
2025-01-01 00:30:00        C049  274.70  1979-01-20     Munich
2025-01-01 01:00:00        C017  380.42  1981-02-23    Cologne
2025-01-01 01:30:00        C049  378.28  1979-01-20     Munich
2025-01-01 02:00:00        C002   13.88  1988-07-03  Frankfurt


In [9]:
#3.1
cust_stats = merged.groupby('Customer_id').agg(
    total_revenue=('Price', 'sum'),
    n_purchases=('Price', 'count'),
    avg_price=('Price','mean')
)
print(cust_stats.head())

             total_revenue  n_purchases   avg_price
Customer_id                                        
C001              13055.75           56  233.138393
C002              15927.80           64  248.871875
C003              14947.34           56  266.916786
C004              15111.43           59  256.125932
C005              16798.61           64  262.478281


In [ ]:
#3.2
merged['Month'] = merged.index.month_name()

# 3.3
city_month = merged.groupby(['City', 'Month']).agg(monthly_revenue=('Price', 'sum'))
print(city_month)

                    monthly_revenue
City      Month                    
Berlin    February         86163.85
          January         101192.17
          March            10474.86
Cologne   February         86781.31
          January          98030.00
          March            11220.64
Frankfurt February         52020.01
          January          61492.29
          March             6698.88
Hamburg   February         42056.17
          January          43988.43
          March             5340.12
Munich    February         67799.84
          January          78726.19
          March             7330.35


In [11]:
pv_city = pd.pivot_table(merged, values="Price", index="City", columns="Month", aggfunc="sum")
print(pv_city)

Month      February    January     March
City                                    
Berlin     86163.85  101192.17  10474.86
Cologne    86781.31   98030.00  11220.64
Frankfurt  52020.01   61492.29   6698.88
Hamburg    42056.17   43988.43   5340.12
Munich     67799.84   78726.19   7330.35


In [ ]:
#4 
df4 = merged.reset_index().set_index(['Customer_id', 'Timestamp']).sort_index()
df4['roll3_price'] = df4.groupby('Customer_id')['Price'].transform(lambda x: x.rolling(3).mean())

df4['price_diff'] = df4['Price'] - df4.groupby('Customer_id')['Price'].shift(1)
print(df4.head(10))

                                  Price   Born_date       City    Month  \
Customer_id Timestamp                                                     
C001        2025-01-01 05:30:00  376.61  1998-12-03  Frankfurt  January   
            2025-01-02 02:00:00   63.23  1998-12-03  Frankfurt  January   
            2025-01-02 10:00:00  108.83  1998-12-03  Frankfurt  January   
            2025-01-05 05:00:00   70.19  1998-12-03  Frankfurt  January   
            2025-01-06 07:00:00  132.80  1998-12-03  Frankfurt  January   
            2025-01-06 07:30:00  338.93  1998-12-03  Frankfurt  January   
            2025-01-06 11:30:00  415.65  1998-12-03  Frankfurt  January   
            2025-01-07 08:00:00  196.76  1998-12-03  Frankfurt  January   
            2025-01-07 19:30:00  382.14  1998-12-03  Frankfurt  January   
            2025-01-10 10:30:00  265.98  1998-12-03  Frankfurt  January   

                                 roll3_price  price_diff  
Customer_id Timestamp                   

In [ ]:
#5.1 
berlin = customer_info.query('City == "Berlin"')[['Customer_id', 'Born_date', 'City']]
df_berlin = df[df['Customer_id'].isin(berlin['Customer_id'])]
df_berlin = df_berlin.reset_index().merge(berlin, on='Customer_id', how='left')

df_berlin = df_berlin.set_index(['Customer_id', 'Timestamp']).sort_index()
df_berlin['roll3_price'] = df_berlin.groupby('Customer_id')['Price'].transform(lambda x: x.rolling(3).mean())
df_berlin['price_diff'] = df_berlin['Price'] - df_berlin.groupby('Customer_id')['Price'].shift(1)
print(df_berlin.head(10))

                                  Price   Born_date    City  roll3_price  \
Customer_id Timestamp                                                      
C013        2025-01-05 02:30:00  474.31  1995-06-12  Berlin          NaN   
            2025-01-06 00:00:00  410.01  1995-06-12  Berlin          NaN   
            2025-01-07 23:00:00  396.90  1995-06-12  Berlin   427.073333   
            2025-01-08 22:00:00   92.70  1995-06-12  Berlin   299.870000   
            2025-01-09 04:00:00  392.28  1995-06-12  Berlin   293.960000   
            2025-01-10 03:00:00   34.44  1995-06-12  Berlin   173.140000   
            2025-01-10 20:30:00  405.55  1995-06-12  Berlin   277.423333   
            2025-01-11 01:30:00  276.82  1995-06-12  Berlin   238.936667   
            2025-01-12 06:30:00  448.38  1995-06-12  Berlin   376.916667   
            2025-01-12 08:30:00  470.52  1995-06-12  Berlin   398.573333   

                                 price_diff  
Customer_id Timestamp                    

In [ ]:
#5.2
df5 = df.reset_index().set_index(['Customer_id','Timestamp']).sort_index()
df5['prev_price'] = df5.groupby('Customer_id')['Price'].shift(1)
df5 = df5[df5['Price'] > 1.2 * df5['prev_price']]   # filtr
df5 = df5.reset_index().merge(customer_info[['Customer_id', 'Born_date', 'City']], on='Customer_id', how='left')
df5 = df5.set_index(['Customer_id','Timestamp']).sort_index()
df5['roll3_price'] = df5.groupby('Customer_id')['Price'].transform(lambda x: x.rolling(3).mean())
df5['price_diff'] = df5['Price'] - df5.groupby('Customer_id')['Price'].shift(1)
print(len(df5))
print(df5.head(10))

1208
                                  Price  prev_price   Born_date       City  \
Customer_id Timestamp                                                        
C001        2025-01-02 10:00:00  108.83       63.23  1998-12-03  Frankfurt   
            2025-01-06 07:00:00  132.80       70.19  1998-12-03  Frankfurt   
            2025-01-06 07:30:00  338.93      132.80  1998-12-03  Frankfurt   
            2025-01-06 11:30:00  415.65      338.93  1998-12-03  Frankfurt   
            2025-01-07 19:30:00  382.14      196.76  1998-12-03  Frankfurt   
            2025-01-11 15:30:00  210.98       17.13  1998-12-03  Frankfurt   
            2025-01-13 00:30:00   56.88       25.93  1998-12-03  Frankfurt   
            2025-01-13 11:30:00  465.75       56.88  1998-12-03  Frankfurt   
            2025-01-13 20:30:00  419.94       30.84  1998-12-03  Frankfurt   
            2025-01-15 23:00:00  331.26      118.97  1998-12-03  Frankfurt   

                                 roll3_price  price_diff  

In [16]:
#6
last_date = merged.index.max()
last_q = merged[merged.index > last_date - pd.Timedelta(days=90)]
big_customers = last_q.groupby('Customer_id').filter(lambda g: g['Price'].sum() > 1000)
print('zakaznici nad 1000:', big_customers['Customer_id'].unique())


zakaznici nad 1000: <StringArray>
['C037', 'C049', 'C017', 'C002', 'C028', 'C023', 'C032', 'C033', 'C001',
 'C019', 'C044', 'C026', 'C006', 'C004', 'C011', 'C038', 'C024', 'C005',
 'C034', 'C022', 'C048', 'C016', 'C009', 'C029', 'C003', 'C020', 'C036',
 'C007', 'C041', 'C040', 'C039', 'C018', 'C025', 'C050', 'C031', 'C030',
 'C042', 'C035', 'C012', 'C008', 'C046', 'C014', 'C021', 'C045', 'C027',
 'C047', 'C043', 'C013', 'C015', 'C010']
Length: 50, dtype: str


In [17]:
merged['price_share'] = merged.groupby("Customer_id")["Price"].transform(lambda x: x / x.sum())
print(merged.head())

                    Customer_id   Price   Born_date       City    Month  \
Timestamp                                                                 
2025-01-01 00:00:00        C037  436.71  1996-09-14     Munich  January   
2025-01-01 00:30:00        C049  274.70  1979-01-20     Munich  January   
2025-01-01 01:00:00        C017  380.42  1981-02-23    Cologne  January   
2025-01-01 01:30:00        C049  378.28  1979-01-20     Munich  January   
2025-01-01 02:00:00        C002   13.88  1988-07-03  Frankfurt  January   

                     price_share  
Timestamp                         
2025-01-01 00:00:00     0.023330  
2025-01-01 00:30:00     0.015159  
2025-01-01 01:00:00     0.022854  
2025-01-01 01:30:00     0.020876  
2025-01-01 02:00:00     0.000871  


In [ ]:
merged['year'] = merged.index.year
merged['month_num'] = merged.index.month
monthly = merged.groupby(['Customer_id', 'year', 'month_num']).agg(revenue=('Price', 'sum'))
monthly['prev_revenue'] = monthly.groupby('Customer_id')['revenue'].shift(1)

monthly['growth_rate'] = (monthly['revenue'] - monthly['prev_revenue']) / monthly['prev_revenue']
print(monthly.head(15))

                            revenue  prev_revenue  growth_rate
Customer_id year month_num                                    
C001        2025 1          7302.10           NaN          NaN
                 2          5204.79       7302.10    -0.287220
                 3           548.86       5204.79    -0.894547
C002        2025 1          7538.37           NaN          NaN
                 2          7838.50       7538.37     0.039814
                 3           550.93       7838.50    -0.929715
C003        2025 1          7133.09           NaN          NaN
                 2          7141.33       7133.09     0.001155
                 3           672.92       7141.33    -0.905771
C004        2025 1          6920.10           NaN          NaN
                 2          7008.12       6920.10     0.012719
                 3          1183.21       7008.12    -0.831166
C005        2025 1          7928.47           NaN          NaN
                 2          7284.34       7928.47    -0

In [ ]:
# 7.2 
df7 = merged.reset_index().set_index(['Customer_id', 'Timestamp']).sort_index()
df7['cum_price'] = df7.groupby('Customer_id')['Price'].transform(lambda x: x.expanding().sum())
print(df7[['Price', 'cum_price']].head(10))

                                  Price  cum_price
Customer_id Timestamp                             
C001        2025-01-01 05:30:00  376.61     376.61
            2025-01-02 02:00:00   63.23     439.84
            2025-01-02 10:00:00  108.83     548.67
            2025-01-05 05:00:00   70.19     618.86
            2025-01-06 07:00:00  132.80     751.66
            2025-01-06 07:30:00  338.93    1090.59
            2025-01-06 11:30:00  415.65    1506.24
            2025-01-07 08:00:00  196.76    1703.00
            2025-01-07 19:30:00  382.14    2085.14
            2025-01-10 10:30:00  265.98    2351.12
